In [14]:
import os
import time
import json as _json
from datetime import datetime

from dotenv import load_dotenv
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

# -------------------------------------------------------------------
# CONSTANTS (UNCHANGED URLS)
# -------------------------------------------------------------------

BASE_URL = "https://www.tijd.be"
BASE_SEARCH_URL = "https://www.tijd.be/search.html"


# -------------------------------------------------------------------
# DRIVER & UTILITIES
# -------------------------------------------------------------------

def create_driver(headless: bool = False) -> webdriver.Chrome:
    options = webdriver.ChromeOptions()
    if headless:
        options.add_argument("--headless=new")

    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")

    # Perf tweak: block some heavy content (safe to remove if it breaks anything)
    prefs = {
        "profile.managed_default_content_settings.images": 2,
        "profile.managed_default_content_settings.stylesheets": 2,
        "profile.managed_default_content_settings.fonts": 2,
    }
    options.add_experimental_option("prefs", prefs)

    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options,
    )


def accept_cookies_if_present(driver, timeout: int = 5):
    """Accept cookie banner if it appears."""
    try:
        wait = WebDriverWait(driver, timeout)
        cookie_button = wait.until(
            EC.element_to_be_clickable((
                By.CSS_SELECTOR,
                "#didomi-notice-agree-button"  # same selector
            ))
        )
        cookie_button.click()
        time.sleep(1)
    except TimeoutException:
        pass

def scroll_until_marker_visible(driver, code, max_scrolls: int = 40, pause: float = 0.3):
    """
    Scrolls down until the marker div for a live-blog post is found in the DOM,
    or until max_scrolls attempts have been made.
    The marker is:
        <div class="live-blog-page__post-scrollid" id="{code}"></div>
    """
    selector = f"div.live-blog-page__post-scrollid[id='{code}']"
    for _ in range(max_scrolls):
        try:
            elem = driver.find_element(By.CSS_SELECTOR, selector)
            return elem
        except NoSuchElementException:
            driver.execute_script("window.scrollBy(0, window.innerHeight);")
            time.sleep(pause)
    return None


# -------------------------------------------------------------------
# LOGIN
# -------------------------------------------------------------------

def login_ft(driver):
    load_dotenv()
    username = os.getenv("DETIJD_USERNAME")
    password = os.getenv("DETIJD_PASSWORD")
    wait = WebDriverWait(driver, 8)

    driver.get(BASE_URL)
    accept_cookies_if_present(driver)

    # login button (same selector)
    login_button = wait.until(EC.element_to_be_clickable((
        By.CSS_SELECTOR,
        "body > div.o-mainapp > header > div.mfn-grid-container > div > div > div:nth-child(2) > button.btn.hidden--below-tablet.trck_sitenav_login"
    ))) 
    login_button.click()

    # username
    email_field = wait.until(EC.presence_of_element_located((
        By.XPATH,
        "//*[@id='username']"
    )))
    email_field.clear()
    email_field.send_keys(username)
    time.sleep(1)

    # next (same selector)
    driver.find_element(
        By.CSS_SELECTOR,
        "body > div > div.mfn-modal > main > section > div > div > div > div.cbe8a9936.cd39274d8 > div > form > div.c14e5e142 > button"
    ).click()
    time.sleep(1)

    # password
    password_field = wait.until(EC.presence_of_element_located((
        By.CSS_SELECTOR,
        "#password"
    )))
    password_field.clear()
    password_field.send_keys(password)
    time.sleep(1)

    # login submit (same selector)
    driver.find_element(
        By.CSS_SELECTOR,
        "body > div > div.mfn-modal > main > section > div > div > div > form > div.c14e5e142 > button"
    ).click()

    # short wait to ensure login completes
    time.sleep(1.5)


# -------------------------------------------------------------------
# SEARCH PAGE
# -------------------------------------------------------------------

def open_search_page(driver, query: str):
    driver.get(BASE_SEARCH_URL)
    accept_cookies_if_present(driver)
    wait = WebDriverWait(driver, 5)

    # query input (same selector)
    query_input = wait.until(EC.presence_of_element_located((
        By.CSS_SELECTOR,
        "#js-searchpageinput"
    )))
    query_input.clear()
    query_input.send_keys(query)
    time.sleep(2)

    # advanced search button (same selector)
    advanced_search_button = wait.until(EC.element_to_be_clickable((
        By.CSS_SELECTOR,
        "#main-content > div.o-hpgrid__row-flex300persist > section > div.o-hpgrid__block-container.o-hpgrid__block-container--nth1 > div > div > div.c-searchpagebox > form > div.c-searchpagebox__actions > button"
    )))
    advanced_search_button.click()
    time.sleep(2)

    # wait for advanced search fields to be visible
    before_date = wait.until(EC.presence_of_element_located((
        By.CSS_SELECTOR,
        "#advancedSearch > div.c-searchpagebox__advscnd > div > div:nth-child(1) > div > div > input[type=text]"
    )))
    before_date.clear()
    before_date.send_keys("01/01/2024")  # unchanged
    time.sleep(2)

    # wait for results list to appear (same selector as we use later)
    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((
                By.CSS_SELECTOR,
                "#\\:r2\\:-tab1-content > ul.list-unstyled > li > a"
            ))
        )
    except TimeoutException:
        # if nothing appears, still return and let URL collector handle it
        pass


# -------------------------------------------------------------------
# ARTICLE SCRAPER
# -------------------------------------------------------------------

def scrape_article(driver, url: str):
    """Scrape a single article page, with a special case for live-blogs."""
    driver.get(url)
    wait = WebDriverWait(driver, 10)

    # ------------------------------------------------------------
    # 1) SPECIAL CASE: live-blog posts based on {code} and marker div
    # ------------------------------------------------------------
    if "live-blog" in url:
        try:
            code = None
            if "#!" in url:
                # e.g. ...html#!890307 -> code = "890307"
                code = url.split("#!", 1)[1]

            if code:
                # Ensure the marker div is actually loaded into the DOM
                marker = scroll_until_marker_visible(driver, code)
                if marker is None:
                    print(f"[LIVE] Could not find marker div for code={code} after scrolling.")
                else:
                    # Optional: center it on screen
                    driver.execute_script(
                        "arguments[0].scrollIntoView({block: 'center'});",
                        marker
                    )

                    # Article directly after the marker:
                    # <div class="live-blog-page__post-scrollid" id="{code}"></div>
                    # <article class="live-blog-post ...">...</article>
                    article = driver.find_element(
                        By.CSS_SELECTOR,
                        f"div.live-blog-page__post-scrollid[id='{code}'] + article[class*='live-blog-post']"
                    )

                    # Title of the blog post
                    title_el = article.find_element(By.CSS_SELECTOR, ".live-blog-post__title")
                    title = title_el.text.strip()

                    # Body paragraphs inside this specific post
                    # (matching your outer HTML: .ac_paragraph.paywalled-content p)
                    paras = article.find_elements(
                        By.CSS_SELECTOR,
                        ".ac_paragraph.paywalled-content p"
                    )
                    content = "\n".join(p.text for p in paras if p.text)

                    # Date of the live-blog (your original selector)
                    date_iso = ""
                    try:
                        date_el = driver.find_element(
                            By.CSS_SELECTOR,
                            ".js-timeToday-date:nth-child(1)"
                        )
                        date_text = date_el.text.strip()
                        for fmt in ("%d/%m/%Y", "%d-%m-%Y"):
                            try:
                                date_iso = datetime.strptime(date_text, fmt).isoformat()
                                break
                            except ValueError:
                                continue
                        if not date_iso:
                            date_iso = date_text
                    except Exception:
                        # fallback: any <time datetime> on the page
                        date_els = driver.find_elements(By.CSS_SELECTOR, "time[datetime]")
                        if date_els:
                            date_iso = date_els[0].get_attribute("datetime") or ""

                    print(f"[LIVE] Scraped blog post: {title}, date: {date_iso}")
                    return {
                        "url": url,
                        "title": title,
                        "date": date_iso,
                        "content": content
                    }

        except Exception as e:
            print(f"[LIVE] Failed to parse live-blog special case for {url} (code={code}): {e}")
            # Fall through to normal article logic below

    # ------------------------------------------------------------
    # 2) NORMAL ARTICLE FLOW (keep your existing logic here)
    # ------------------------------------------------------------

    # title (your latest version uses just h1)
    try:
        title_el = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "h1")))
    except TimeoutException:
        # fallback to your older specific selector
        try:
            title_el = wait.until(EC.presence_of_element_located((
                By.CSS_SELECTOR,
                "#main-content > div > div.o-articlehead > header > h1"
            )))
        except TimeoutException:
            return None

    title = title_el.text.strip()

    # publication date
    date_iso = ""
    # 1) generic <time datetime>
    date_els = driver.find_elements(By.CSS_SELECTOR, "time[datetime]")
    if date_els:
        date_iso = date_els[0].get_attribute("datetime") or ""

    # 2) fallback to your older long selector if nothing
    if not date_iso:
        date_els = driver.find_elements(
            By.CSS_SELECTOR,
            "#main-content > div > div.o-articlebody > div.o-articlebody-left > "
            "div.c-articlehead__detail.paywalled-content > "
            "div.c-articlehead__detail__dates > div > time"
        )
        if date_els:
            date_iso = date_els[0].get_attribute("datetime") or ""

    # content
    content = ""
    try:
        # your structured article content container
        article_body = wait.until(EC.presence_of_element_located((
            By.CSS_SELECTOR,
            "#main-content > div > div.o-articlebody > div.o-articlebody-left > div.c-articlecontents"
        )))
        paras = article_body.find_elements(By.CSS_SELECTOR, "p")
        content = "\n".join(p.text for p in paras if p.text)
    except TimeoutException:
        # older fallback: .ac_paragraph > p
        paras = driver.find_elements(By.CSS_SELECTOR, ".ac_paragraph > p")
        content = "\n".join(p.text for p in paras if p.text)

    print(f"Scraped article: {title}, date: {date_iso}")
    return {
        "url": url,
        "title": title,
        "date": date_iso,
        "content": content,
    }


# -------------------------------------------------------------------
# PHASE 1: COLLECT ONLY URLS FROM SEARCH RESULTS
# -------------------------------------------------------------------

def collect_result_urls(driver, max_pages: int | None = None) -> list[str]:
    """
    Collect all article URLs from paginated search results.
    Uses exactly the same selectors as before, but does NOT scrape content.
    """
    urls_collected: list[str] = []
    seen_urls: set[str] = set()
    next_page = True
    page_count = 0
    wait = WebDriverWait(driver, 10)

    while next_page:
        page_count += 1
        if max_pages is not None and page_count > max_pages:
            break

        # wait for links to appear on the results tab (same selector)
        try:
            wait.until(
                EC.presence_of_all_elements_located((
                    By.CSS_SELECTOR,
                    "#\\:r2\\:-tab1-content > ul.list-unstyled > li > a"
                ))
            )
        except TimeoutException:
            print("[INFO] No more search results found.")
            break

        page_links = driver.find_elements(
            By.CSS_SELECTOR,
            "#\\:r2\\:-tab1-content > ul.list-unstyled > li > a"
        )

        page_urls = [
            lnk.get_attribute("href")
            for lnk in page_links
            if lnk.get_attribute("href")
        ]

        for url in page_urls:
            if url in seen_urls:
                continue
            seen_urls.add(url)
            urls_collected.append(url)

        print(f"[INFO] Collected {len(page_urls)} URLs on page {page_count} "
              f"(total unique so far: {len(urls_collected)})")

        # handle pagination button (same selector you provided)
        try:
            next_button = driver.find_element(
                By.CSS_SELECTOR,
                "#\:r2\:-tab1-content > ul.list-flexbetween.c-page-navigation > li:nth-child(3) > button"
            )
            btn_class = next_button.get_attribute("class") or ""
            if "btn btn-smallpadding btn-global-2 disabled" in btn_class:
                next_page = False
            else:
                next_button.click()
                # wait a bit for next page content
                time.sleep(1)
        except Exception:
            next_page = False

    return urls_collected


# -------------------------------------------------------------------
# MAIN SCRAPER LOOP (PHASE 1: URLs, PHASE 2: ARTICLES)
# -------------------------------------------------------------------

def run_scrape(headless: bool = False, max_pages: int = 12, pause: float = 1.5):
    driver = create_driver(headless=headless)

    # your query list (unchanged)
    query_list = [
        "Zilver bedrijven",
        "Koper bedrijven",
        "Platinum",
        "Palladium",
        "Industriële Metalen", 
        "Edelmetalen"
    ]

    try:
        login_ft(driver)

        # ----------
        # goud DONE: query_list[0:1]
        # Zilver: query_list[1:2]
        # platina DONE: query_list[2:3]
        # koper DONE: query_list[3:4]
        # Mijnen DONE: query_list[4:5]
        # mijnbedrijven DONE: query_list[5:6]
        # goud bedrijven DONE: query_list[6:7]
        # edel metalen DONE: query_list[7:8]
        # ----------


        for q in query_list[3:]:
            print(f"\n[INFO] Searching for query: {q}")
            open_search_page(driver, query=q)
            time.sleep(1)

            # ---- PHASE 1: collect all URLs for this query ----
            urls = collect_result_urls(driver, max_pages=max_pages)
            print(f"[INFO] Collected {len(urls)} URLs for query '{q}'")

            # ---- PHASE 2: scrape each URL ----
            articles = []
            for idx, url in enumerate(urls, start=1):
                print(f"[INFO] ({idx}/{len(urls)}) Scraping article: {url}")
                art = scrape_article(driver, url)
                if art:
                    articles.append(art)
                # small pause to be polite
                time.sleep(0.5)

            print(f"[INFO] Scraped {len(articles)} articles for query '{q}'")

            # save per query (same folder, file pattern)
            safe_q = q.replace(" ", "_")
            save_jsonl(articles, f"./Experts_articles/DETIJD_{safe_q}.jsonl")

            # Reset to base URL before next query
            driver.get(BASE_URL)
            time.sleep(pause)

    finally:
        driver.quit()


# -------------------------------------------------------------------
# OUTPUT
# -------------------------------------------------------------------

def save_jsonl(records, path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for rec in records:
            f.write(_json.dumps(rec, ensure_ascii=False) + "\n")


def main():
    # you can tweak max_pages as needed
    run_scrape(headless=False, max_pages=50)


if __name__ == "__main__":
    main()

<>:389: SyntaxWarning: invalid escape sequence '\:'
<>:389: SyntaxWarning: invalid escape sequence '\:'
/var/folders/24/3ptrhbtn7x7cvnh_pyprvkrh0000gn/T/ipykernel_50791/3949532982.py:389: SyntaxWarning: invalid escape sequence '\:'
  "#\:r2\:-tab1-content > ul.list-flexbetween.c-page-navigation > li:nth-child(3) > button"



[INFO] Searching for query: Palladium
[INFO] Collected 20 URLs on page 1 (total unique so far: 20)
[INFO] Collected 12 URLs on page 2 (total unique so far: 32)
[INFO] Collected 32 URLs for query 'Palladium'
[INFO] (1/32) Scraping article: https://www.tijd.be/netto/dossiers/spaar-en-beleggingsgids/waarom-2026-een-nieuw-glansjaar-kan-worden-voor-goud/10639858.html
Scraped article: Waarom 2026 een nieuw glansjaar kan worden voor goud, date: 2025-12-11 05:00
[INFO] (2/32) Scraping article: https://www.tijd.be/netto/dossiers/spaar-en-beleggingsgids/alles-wat-u-moet-weten-over-de-meerwaardebelasting/10639894.html
Scraped article: Alles wat u moet weten over de meerwaardebelasting, date: 2025-12-11 05:00
[INFO] (3/32) Scraping article: https://www.tijd.be/dossiers/meerwaardebelasting/meer-duidelijkheid-over-meerwaardebelasting-dit-moet-u-weten/10639399.html
Scraped article: Meer duidelijkheid over meerwaardebelasting: dit moet u weten, date: 2025-12-08 15:28
[INFO] (4/32) Scraping article: h

In [ ]:
#\38 90305
#\!890305